# Filtered 1D-Nanotube Template Forensics

Simple forensic analysis of the **filtered** pinning database
`nanotube_templates.npz`, built by `build_templates.py --filter` (quality gates in
`filter_templates.py`). A sibling of `comp_models/Analysis/nanotube_rtheta_forensics.ipynb`,
but it reads the compact CSR npz and reuses the exact metric/geometry helpers from
`filter_templates.py`, so the analysis and the DB filter always agree.

Every template here has already passed the 4 gates (contacts, hollow core, atom-count,
peaked ρ) — this notebook characterizes the survivors. **Writes nothing to disk.**

In [ ]:
# --- dependencies (run once) -------------------------------------------------
%pip install -q numpy pandas matplotlib scipy
import numpy, pandas, matplotlib, scipy
for m in (numpy, pandas, matplotlib, scipy):
    print(f"{m.__name__:12s} {m.__version__}")

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

HERE = Path.cwd()                     # run from NTGENS/data/nano_1D
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
import filter_templates as ft         # metrics, gates, geometry, draw_tube_3d

NPZ  = HERE / "nanotube_templates.npz"
SEED = 0
rng  = np.random.default_rng(SEED)

mpl.rcParams.update({
    "font.size": 12, "figure.dpi": 110, "figure.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
    "axes.spines.right": False, "axes.titleweight": "bold",
})
ACCENT = ["#43AA8B", "#F8961E", "#F94144", "#277DA1"]
print("npz exists:", NPZ.exists(), "| thresholds:",
      dict(MIN_CONTACT=ft.MIN_CONTACT, MIN_RMIN=ft.MIN_RMIN,
           NATM=(ft.NATM_MIN, ft.NATM_MAX), PEAK_RATIO=ft.PEAK_RATIO))

## 1 · Load the filtered npz + compute metrics

In [ ]:
# Load the filtered CSR npz -> per-template records, and compute metrics for each.
templates = ft.load_npz_records(NPZ)
M = pd.DataFrame([dict(**ft.template_metrics(t["numbers"], t["frac"], t["cell"]),
                       source={0: "synthetic", 1: "real", -1: "untagged"}[t["source"]])
                  for t in templates])
print(f"Loaded {len(templates):,} filtered templates")
print("source split:", M.source.value_counts().to_dict())
display(M[["nsites", "r_min", "r_max", "min_nn", "peak_ratio"]].describe().round(2))

## 2 · Sanity check — all survivors clear the gates

In [ ]:
# Sanity: every template should already clear all 4 gates.
passes = np.array([ft.passes_filter(ft.template_metrics(t["numbers"], t["frac"], t["cell"]))
                   for t in templates])
print(f"pass all gates: {passes.sum():,} / {len(passes):,} "
      f"({100*passes.mean():.1f}%)  -> expect 100%")

## 3 · Population distributions

In [ ]:
# Population distributions of the gate metrics (survivors only).
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
specs = [("nsites", "atoms / cell", ft.NATM_MIN),
         ("r_min", "r$_{min}$ (Å)  [inner hollow radius]", ft.MIN_RMIN),
         ("r_max", "r$_{max}$ (Å)  [outer wall radius]", None),
         ("min_nn", "min contact (Å)", ft.MIN_CONTACT),
         ("peak_ratio", r"$\rho_{peak}/\bar{\rho}$", ft.PEAK_RATIO)]
for a, (col, lab, thr) in zip(ax.flat, specs):
    a.hist(M[col].dropna(), bins=40, color=ACCENT[3], edgecolor="white")
    if thr is not None:
        a.axvline(thr, color=ACCENT[2], ls="--", lw=2, label=f"gate = {thr}")
        a.legend()
    a.set_xlabel(lab); a.set_ylabel("templates")
# source split pie
a = ax.flat[5]; vc = M.source.value_counts()
a.pie(vc.values, labels=vc.index, autopct="%1.0f%%",
      colors=[ACCENT[0], ACCENT[1], ACCENT[2]][:len(vc)])
a.set_title("provenance"); a.grid(False)
fig.suptitle("Filtered template population — gate-metric distributions", fontweight="bold")
plt.tight_layout(); plt.show()

## 4 · Single template inspection — six-view geometry

Set `template_idx` below to inspect any one filtered template (`0 .. len(templates)-1`).

In [ ]:
# One filtered template, six-view geometry (Perspective / Front / Side / Top /
# Down tube axis / Axis profile), with r_min (blue) / r_max (red) cylinders about
# the detected tube axis -- same view layout as the generation-notebook validation cell.
template_idx = 0   # <-- change this to inspect a different template (0 .. len(templates)-1)

t = templates[template_idx]
cart, cell = t["cart"], t["cell"]
els = ft.symbols_of(t["numbers"]); uniq = sorted(set(els))
cmap = {e: ft._OKABE[i % len(ft._OKABE)] for i, e in enumerate(uniq)}
m = ft.template_metrics(t["numbers"], t["frac"], t["cell"])
formula = "".join(uniq)
src_label = {0: "synthetic", 1: "real", -1: "untagged"}[t["source"]]

print(f"template_idx = {template_idx}   ─ {formula}  (N={m['nsites']}, {src_label})")
print(f"  r_min={m['r_min']:.2f}  r_max={m['r_max']:.2f} Å   "
      f"min contact={m['min_nn']:.2f} Å   ρ_peak/ρ̄={m['peak_ratio']:.2f}")

# --- Cylindrical frame about the detected tube axis (centroid + in-plane basis) ---
ax_i = ft.detect_tube_axis(t["frac"])
a_hat = cell[ax_i] / np.linalg.norm(cell[ax_i])
z = cart @ a_hat
perp = cart - np.outer(z, a_hat)
other = [k for k in range(3) if k != ax_i]
e1 = cell[other[0]] - (cell[other[0]] @ a_hat) * a_hat
e1 = e1 / np.linalg.norm(e1)
e2 = np.cross(a_hat, e1)
ctr = perp.mean(0)
band = dict(ctr=ctr, a=a_hat, e1=e1, e2=e2, r_lo=m["r_min"], r_hi=m["r_max"])

def cyl_rings(band, r, cart, n_rings=6, n_theta=60):
    """Coaxial cylinder of radius r spanning the atoms' axial range (list of rings)."""
    a, ctr, e1, e2 = band["a"], band["ctr"], band["e1"], band["e2"]
    zc = cart @ a
    th = np.linspace(0, 2 * np.pi, n_theta)
    ring = lambda z: (ctr + z * a) + r * np.outer(np.cos(th), e1) + r * np.outer(np.sin(th), e2)
    return [ring(z) for z in np.linspace(zc.min(), zc.max(), n_rings)]

def axis_angles(a):
    """(elev, azim) that look DOWN the tube axis -> the tube's cross-section."""
    return np.degrees(np.arcsin(np.clip(a[2], -1, 1))), np.degrees(np.arctan2(a[1], a[0]))

dax = axis_angles(band["a"])
prof = (0.0, np.degrees(np.arctan2(band["a"][1], band["a"][0])) + 90.0)
views = [("Perspective", 22, -60), ("Front (xz)", 0, -90), ("Side (yz)", 0, 0),
         ("Top (xy)", 90, -90), ("Down tube axis", *dax), ("Axis profile", *prof)]

# Common cube so every panel has TRUE equal aspect (a tube must look like a tube).
mid = cart.mean(0)
half = float((cart.max(0) - cart.min(0)).max()) / 2 + 1.0

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
fig = plt.figure(figsize=(16, 9))
for k, (title, elev, azim) in enumerate(views):
    a = fig.add_subplot(2, 3, k + 1, projection="3d")
    for e in uniq:
        mm = els == e
        a.scatter(cart[mm, 0], cart[mm, 1], cart[mm, 2], s=55, color=cmap[e],
                  edgecolors="k", linewidths=0.3, label=e)
    for r, col in [(band["r_lo"], "#277DA1"), (band["r_hi"], "#F94144")]:
        if r <= 0:
            continue
        for pts in cyl_rings(band, r, cart):
            a.plot(pts[:, 0], pts[:, 1], pts[:, 2], color=col, lw=0.8, alpha=0.45)
    a.set_title(title, fontsize=10)
    a.view_init(elev=elev, azim=azim)
    a.set_xlim(mid[0] - half, mid[0] + half)
    a.set_ylim(mid[1] - half, mid[1] + half)
    a.set_zlim(mid[2] - half, mid[2] + half)
    a.set_xticklabels([]); a.set_yticklabels([]); a.set_zticklabels([])
    try:
        a.set_box_aspect((1, 1, 1))
    except Exception:
        pass
    if k == 0:
        a.legend(fontsize=7, loc="upper left", framealpha=0.6)

fig.suptitle(f"{formula}  —  {m['nsites']} atoms  —  filter-gate shell "
             f"(blue=r_min 5th-pct, red=r_max 95th-pct)",
             y=0.99, fontweight="bold")
plt.tight_layout(); plt.show()